In [1]:
import json
import glob
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

/Users/vladis/Desktop/kpi_labs/rhetoric-proximity/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 1. ЗАВАНТАЖЕННЯ ДАНИХ З JSON
# ==========================================
print("Завантаження даних...")

all_data = []

json_files = glob.glob("data/*.json")

records_count = 0
for file in json_files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Якщо файл містить список — обробляємо кожен елемент
    if isinstance(data, list):
        records = data
    else:
        records = [data]

    records_count += len(records)

    for item in records:
        person = item.get("person", "Unknown")

        for post in item.get("posts", []):
            annotation = post.get("annotation")
            if not annotation or "axes" not in annotation:
                continue

            axes = annotation["axes"]

            row = {
                "person": person,
                "text": post.get("text", "").strip(),
                "militarism": axes.get("militarism", 0.0),
                "national_identity": axes.get("national_identity", 0.0),
                "traditionalism": axes.get("traditionalism", 0.0),
                "statism": axes.get("statism", 0.0),
                "populism": axes.get("populism", 0.0)
            }

            all_data.append(row)

df = pd.DataFrame(all_data)
print(f"Зібрано {len(df)} текстів з {records_count}")

Завантаження даних...
Зібрано 5119 текстів з 8753


In [3]:
# ==========================================
# 2. ГЕНЕРАЦІЯ ЕМБЕДИНГІВ
# ==========================================
print("Завантаження мовної моделі для ембедингів...")
# Використовуємо LaBSE або багатомовний MPNet (чудово працюють з українською)
# 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2' - гарний баланс швидкості та якості
model_name = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
encoder = SentenceTransformer(model_name)

print("Векторизація текстів (це може зайняти хвилину-дві)...")
# Перетворюємо тексти на числові вектори (матриця X)
X = encoder.encode(df['text'].tolist(), show_progress_bar=True)

Завантаження мовної моделі для ембедингів...
Векторизація текстів (це може зайняти хвилину-дві)...


Batches: 100%|██████████| 160/160 [08:18<00:00,  3.11s/it]


In [4]:
# ==========================================
# 3. ПІДГОТОВКА ДО ТРЕНУВАННЯ
# ==========================================
target_cols = ['militarism', 'national_identity', 'traditionalism', 'statism', 'populism']
y = df[target_cols].values

# Розбиваємо дані на тренувальну та тестову вибірки (80% на 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Розмір тренувальної вибірки: {X_train.shape[0]}")
print(f"Розмір тестової вибірки: {X_test.shape[0]}")

Розмір тренувальної вибірки: 4095
Розмір тестової вибірки: 1024


In [5]:
# ==========================================
# 4. ТРЕНУВАННЯ МОДЕЛІ
# ==========================================
print("Тренування моделі Multi-Output Regression...")
# Використовуємо Ridge Regression (лінійна регресія з L2-регуляризацією, щоб уникнути перенавчання)
base_estimator = Ridge(alpha=1.0)
multi_regressor = MultiOutputRegressor(base_estimator)

multi_regressor.fit(X_train, y_train)

Тренування моделі Multi-Output Regression...


,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.,Ridge()
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be

In [6]:
# ==========================================
# 5. ОЦІНКА ЯКОСТІ
# ==========================================
print("Оцінка на тестовій вибірці...")
y_pred = multi_regressor.predict(X_test)

# Обрізаємо передбачення, щоб вони не виходили за межі [0, 1]
y_pred = np.clip(y_pred, 0.0, 1.0)

# Рахуємо середню абсолютну помилку (MAE) загалом і по кожній осі
mae_total = mean_absolute_error(y_test, y_pred)
print(f"\nЗагальна середня помилка (MAE): {mae_total:.4f}")
print("Похибка по кожній осі:")

for i, col in enumerate(target_cols):
    mae_col = mean_absolute_error(y_test[:, i], y_pred[:, i])
    print(f" - {col}: {mae_col:.4f}")

# ==========================================
# 6. ТЕСТУВАННЯ НА НОВИХ ДАНИХ (ІНФЕРЕНС)
# ==========================================
print("\n--- Демонстрація роботи на нових текстах ---")
new_texts = [
    "Ворог має бути знищений! Наша армія найсильніша, і ми переможемо будь-якою ціною.",
    "Влада знову краде гроші з бюджету, поки прості українці страждають через відключення світла."
]

new_embeddings = encoder.encode(new_texts)
predictions = multi_regressor.predict(new_embeddings)
predictions = np.clip(predictions, 0.0, 1.0)

for text, pred in zip(new_texts, predictions):
    print(f"\nТекст: {text}")
    for col, value in zip(target_cols, pred):
        print(f"  {col}: {value:.2f}")

Оцінка на тестовій вибірці...

Загальна середня помилка (MAE): 0.1022
Похибка по кожній осі:
 - militarism: 0.1387
 - national_identity: 0.1175
 - traditionalism: 0.0411
 - statism: 0.1192
 - populism: 0.0945

--- Демонстрація роботи на нових текстах ---

Текст: Ворог має бути знищений! Наша армія найсильніша, і ми переможемо будь-якою ціною.
  militarism: 1.00
  national_identity: 0.67
  traditionalism: 0.04
  statism: 0.15
  populism: 0.00

Текст: Влада знову краде гроші з бюджету, поки прості українці страждають через відключення світла.
  militarism: 0.37
  national_identity: 0.56
  traditionalism: 0.04
  statism: 0.50
  populism: 0.44


In [ ]:
import joblib

print("Збереження моделі...")
joblib.dump(multi_regressor, 'political_model.pkl')
print("Модель збережено у файл 'political_model.pkl'!")

Модель успішно збережена у файл 'political_model.joblib'
